# Tahap 1: Membangun Case Base
**SubCPMK-3 — Penalaran Komputer | NIM: 202310370311358**

Tujuan: Mengunduh dan melakukan preprocessing teks putusan pidana **Narkotika & Psikotropika** (≥30 dokumen) dari Direktori Putusan Mahkamah Agung RI.

In [ ]:
# Install dependencies (jalankan jika di Google Colab)
# !pip install requests beautifulsoup4 pdfminer.six tqdm pandas

In [ ]:
import os
import re
import time
import logging
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm

# Setup direktori
RAW_DIR = Path('../data/raw')
LOG_DIR = Path('../logs')
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Setup logging
logging.basicConfig(
    filename=LOG_DIR / 'cleaning.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
print('Setup selesai.')

## 1.1 Seleksi & Unduh Putusan dari Direktori MA RI

Sumber: https://putusan3.mahkamahagung.go.id/

> **Catatan:** Jika scraping otomatis terblokir, gunakan Metode B (konversi PDF manual).

In [ ]:
# === METODE A: Scraping otomatis dari direktori MA ===

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'
}
BASE_URL = 'https://putusan3.mahkamahagung.go.id'
SEARCH_URL = (
    'https://putusan3.mahkamahagung.go.id/search.html'
    '?q=narkotika+psikotropika'
    '&cat=2251c1d13f0044e2007a7c4c09ac85ce'
    '&page={page}'
)

def get_case_links(page=1):
    """Ambil daftar link putusan dari halaman pencarian MA."""
    url = SEARCH_URL.format(page=page)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        links = []
        for a in soup.select('a[href*="/direktori/putusan/"]'):
            href = a.get('href', '')
            if href and href not in links:
                full = BASE_URL + href if href.startswith('/') else href
                links.append(full)
        return list(dict.fromkeys(links))
    except Exception as e:
        logging.error(f'Gagal ambil halaman {page}: {e}')
        print(f'[ERROR] Halaman {page}: {e}')
        return []

# Kumpulkan link dari beberapa halaman
all_links = []
for page in range(1, 6):
    links = get_case_links(page)
    all_links.extend(links)
    print(f'Halaman {page}: {len(links)} link ditemukan')
    time.sleep(1.5)

all_links = list(dict.fromkeys(all_links))
print(f'\nTotal link unik terkumpul: {len(all_links)}')

In [ ]:
# === METODE B: Konversi PDF yang diunduh manual ===
# Letakkan file PDF di folder data/raw/pdfs/, lalu jalankan sel ini

from pdfminer.high_level import extract_text as pdf_extract

PDF_DIR = Path('../data/raw/pdfs')
PDF_DIR.mkdir(parents=True, exist_ok=True)

def pdf_to_text(pdf_path):
    """Konversi satu file PDF ke string teks."""
    try:
        text = pdf_extract(str(pdf_path))
        return text
    except Exception as e:
        logging.error(f'Gagal konversi {pdf_path.name}: {e}')
        return None

print(f'Letakkan PDF di: {PDF_DIR.resolve()}')
print(f'PDF tersedia   : {len(list(PDF_DIR.glob("*.pdf")))} file')

## 1.2 Ekstraksi Teks dari HTML

In [ ]:
def extract_text_from_url(url):
    """Ekstrak teks putusan dari halaman HTML direktori MA."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        # Coba beberapa selektor konten utama
        content = (
            soup.find('div', {'id': 'content-wrapper'}) or
            soup.find('div', class_='tab-content') or
            soup.find('div', class_='putusanmenu') or
            soup.find('body')
        )
        return content.get_text(separator='\n') if content else ''
    except Exception as e:
        logging.error(f'Gagal ekstrak {url}: {e}')
        return None

## 1.3 Pembersihan Teks

In [ ]:
def clean_text(text):
    """
    Bersihkan teks putusan MA:
    - Hapus header/footer (nomor halaman, watermark, disclaimer)
    - Normalisasi whitespace dan karakter
    """
    if not text:
        return ''

    # Hapus nomor halaman
    text = re.sub(r'-\s*\d+\s*-', '', text)
    text = re.sub(r'Hal\.?\s*\d+\s*(dari|of)\s*\d+', '', text, flags=re.IGNORECASE)

    # Hapus header/footer standar MA
    for pattern in [
        r'Disclaimer[^\n]*\n?',
        r'Kepaniteraan[^\n]*\n?',
        r'Mahkamah Agung Republik Indonesia[^\n]*\n?',
        r'putusan\.mahkamahagung\.go\.id',
        r'www\.mahkamahagung\.go\.id',
        r'DIREKTORI PUTUSAN[^\n]*\n?',
        r'\f',
    ]:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # Normalisasi whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Uji fungsi
test = "  Mahkamah Agung Republik Indonesia\n\n- 1 -\nPUTUSAN\nNomor 123/Pid.Sus/2023  "
print(repr(clean_text(test)))

## 1.4 Validasi Keutuhan Teks

In [ ]:
def validate_text(text, min_words=200):
    """
    Validasi teks: dianggap valid jika
    - Minimal min_words kata
    - Mengandung kata kunci putusan
    Mengembalikan (is_valid, word_count)
    """
    wc = len(text.split())
    keywords = ['putusan', 'terdakwa', 'dakwaan', 'mengadili', 'pasal']
    has_kw = any(kw in text.lower() for kw in keywords)
    return wc >= min_words and has_kw, wc

## 1.5 Pipeline Utama: Download, Bersihkan, Validasi, Simpan

In [ ]:
TARGET = 30
saved, skipped = 0, 0
metadata_log = []

for url in tqdm(all_links[:60], desc='Mengunduh putusan'):
    if saved >= TARGET + 5:
        break

    raw = extract_text_from_url(url)
    if not raw:
        skipped += 1
        continue

    cleaned = clean_text(raw)
    is_valid, wc = validate_text(cleaned)

    if not is_valid:
        logging.warning(f'SKIP: {url} | words={wc}')
        skipped += 1
        continue

    case_id = f'case_{saved+1:03d}'
    (RAW_DIR / f'{case_id}.txt').write_text(cleaned, encoding='utf-8')
    metadata_log.append({'case_id': case_id, 'url': url, 'word_count': wc})
    logging.info(f'SAVED: {case_id} | words={wc} | {url}')
    saved += 1
    time.sleep(0.5)

print(f'\n=== Hasil Tahap 1 ===')
print(f'Tersimpan : {saved} dokumen')
print(f'Dilewati  : {skipped} dokumen')

## 1.6 Verifikasi & Statistik Akhir

In [ ]:
txt_files = list(RAW_DIR.glob('*.txt'))
print(f'Total file .txt di data/raw/: {len(txt_files)}')
assert len(txt_files) >= 30, f'KURANG! Hanya {len(txt_files)}/30 dokumen'

df_log = pd.DataFrame(metadata_log)
if not df_log.empty:
    print(f"Rata-rata kata : {df_log['word_count'].mean():.0f}")
    print(f"Min kata       : {df_log['word_count'].min()}")
    print(f"Max kata       : {df_log['word_count'].max()}")
    df_log.to_csv('../logs/download_log.csv', index=False)
    display(df_log.head())

print('\n>>> Tahap 1 SELESAI. Lanjut ke 02_representation.ipynb')